In [ ]:
%matplotlib qt

import serial
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import math
import csv
import time
import numpy as np

PORT = '/dev/cu.usbmodem5B420199761'
BAUD_RATE = 921600

# --- Time settings (Adjustable) ---
WAIT_SECONDS = 10       # Countdown time before recording (seconds)
# RECORD_MINUTES = 1    # Recording duration (minutes)
# RECORD_SECONDS = RECORD_MINUTES * 60
RECORD_SECONDS = 30

# Close the port if it's already open
try:
    ser.close()
except:
    pass

try:
    ser = serial.Serial(PORT, BAUD_RATE)
    print(f"Successfully connected to {PORT}! Heatmap is running...")
except Exception as e:
    print(f"Connection failed: {e}")
    raise RuntimeError("Execution stopped: Port is not available. Please check and run again.") 

# Create a CSV file and write the header
csv_filename = f"./experiment/csi_exp_{int(time.time())}.csv"
with open(csv_filename, 'w', newline='') as f:
    writer = csv.writer(f)
    headers = ["Timestamp", "RSSI"] + [f"Sub_{i}" for i in range(64)]
    writer.writerow(headers)

# Set MAX_FRAMES to 500 (5 seconds of past data at 100Hz)
MAX_FRAMES = 500
csi_matrix = np.zeros((64, MAX_FRAMES))

fig, ax = plt.subplots(figsize=(10, 5))
cax = ax.imshow(csi_matrix, aspect='auto', cmap='jet', origin='lower', vmin=0, vmax=30)
fig.colorbar(cax, ax=ax, label='Amplitude')

ax.set_ylabel('Subcarrier Index (0-63)')
ax.set_xlabel('Time (Frames)')

# --- Variables for controlling time and state ---
app_state = "COUNTDOWN"
start_time = time.time()
record_start_time = 0

def update(frame):
    global csi_matrix, app_state, start_time, record_start_time
    
    current_time = time.time()
    
    # Mode 1: Countdown before recording
    if app_state == "COUNTDOWN":
        elapsed = current_time - start_time
        remaining = WAIT_SECONDS - elapsed
        
        if remaining > 0:
            ax.set_title(f' [Get ready...] Recording starts in {remaining:.1f} seconds')
            if ser.is_open:
                ser.reset_input_buffer() # Clear garbage data in the buffer while waiting
            return cax,
        else:
            app_state = "RECORDING"
            record_start_time = time.time()
            print(f"\n--- Started recording data for {RECORD_MINUTES} minutes! ---")
            
    # Mode 2: Recording data to CSV
    elif app_state == "RECORDING":
        elapsed_record = current_time - record_start_time
        remaining_record = RECORD_SECONDS - elapsed_record
        
        # Check if the time is up
        if remaining_record <= 0:
            app_state = "DONE"
            print(f"\n✅ Recording completed for {RECORD_MINUTES} minutes. Stopping automatically.")
            if ser.is_open:
                ser.close()      # Close the serial port
            plt.close(fig)       # Close the plot window
            return cax,
            
        # Update the plot title to show remaining time
        mins, secs = divmod(remaining_record, 60)
        ax.set_title(f' [Recording Heatmap...] Time remaining: {int(mins)} min {secs:.1f} sec')
        
        # Read data and plot as usual
        latest_amplitude = None
        while ser.is_open and ser.in_waiting > 0:
            raw_line = ser.readline().decode('utf-8', errors='ignore').strip()
            if raw_line.startswith("CSI_DATA"):
                try:
                    parts = raw_line.split(',')
                    rssi = int(parts[1]) 
                    iq_data = [int(x) for x in parts[2:]]
                    
                    amplitude = [
                        math.sqrt(iq_data[i]**2 + iq_data[i+1]**2) 
                        for i in range(0, len(iq_data)-1, 2)
                    ]
                    
                    if len(amplitude) >= 64:
                        with open(csv_filename, 'a', newline='') as f:
                            writer = csv.writer(f)
                            writer.writerow([time.time(), rssi] + amplitude[:64])
                        
                        latest_amplitude = amplitude[:64]
                except Exception:
                    pass
                    
        if latest_amplitude is not None:
            csi_matrix[:, :-1] = csi_matrix[:, 1:]
            csi_matrix[:, -1] = latest_amplitude
            cax.set_data(csi_matrix)

    return cax,

# Set interval=10ms (The plot will try to update as fast as possible)
ani = animation.FuncAnimation(fig, update, interval=10, blit=False, cache_frame_data=False)
plt.show()

Successfully connected to /dev/cu.usbmodem5B420199761! Heatmap is running...


Traceback (most recent call last):
  File "/Users/phanlop/RMUTT/Pre-Project/github/AI-Powered-3D-Fall-Detection-System-using-WiFi-Sensing/.venv/lib/python3.10/site-packages/matplotlib/backend_bases.py", line 1152, in _on_timer
    ret = func(*args, **kwargs)
  File "/Users/phanlop/RMUTT/Pre-Project/github/AI-Powered-3D-Fall-Detection-System-using-WiFi-Sensing/.venv/lib/python3.10/site-packages/matplotlib/animation.py", line 1462, in _step
    still_going = super()._step(*args)
  File "/Users/phanlop/RMUTT/Pre-Project/github/AI-Powered-3D-Fall-Detection-System-using-WiFi-Sensing/.venv/lib/python3.10/site-packages/matplotlib/animation.py", line 1150, in _step
    self._draw_next_frame(framedata, self._blit)
  File "/Users/phanlop/RMUTT/Pre-Project/github/AI-Powered-3D-Fall-Detection-System-using-WiFi-Sensing/.venv/lib/python3.10/site-packages/matplotlib/animation.py", line 1169, in _draw_next_frame
    self._draw_frame(framedata)
  File "/Users/phanlop/RMUTT/Pre-Project/github/AI-Powered

In [2]:
# รันเซลล์นี้เพื่อปิดการเชื่อมต่อพอร์ต USB อย่างปลอดภัย
try:
    ser.close()
    print("ปิดพอร์ตเรียบร้อยแล้ว!")
except:
    print("ไม่มีพอร์ตที่เปิดอยู่")

ปิดพอร์ตเรียบร้อยแล้ว!
